# Problem statement.
### Build A Machine learning model to accuretely predict whether or not the patients in the dataset have diabetes or not?

## 2. Data Collection

### Import Necessary libraries.

In [82]:
import pandas as pd
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Dropout
from sklearn.model_selection import GridSearchCV,KFold
## I am building a Neural Network model using keras functional API, and i am using GridsearchCV with kerasclassifer to find tune.
from scikeras.wrappers import KerasClassifier
import numpy as np

import warnings
warnings.filterwarnings("ignore")

### Import data set.

In [83]:
diabetes_data = pd.read_csv("diabetes.csv")
diabetes_data

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


## 3. Data Understanding

In [84]:
diabetes_data.shape

(768, 9)

In [85]:
diabetes_data.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [86]:
diabetes_data.dtypes

Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

In [87]:
diabetes_data.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


## 4. Data Preparation

In [88]:
X = diabetes_data.drop(labels='Outcome',axis = 1)
y = diabetes_data[["Outcome"]]

In [89]:
X.shape

(768, 8)

In [90]:
y.shape

(768, 1)

## 5. Model Building

In [91]:
X_train,X_test,y_train,y_test =  train_test_split(X,y,test_size=0.20,random_state=12,stratify=y)

In [92]:
X_train.shape,y_train.shape

((614, 8), (614, 1))

In [93]:
X_test.shape,y_test.shape

((154, 8), (154, 1))

### Alert: Here's where you Need to apply Standarization.

In [94]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

## Hyperparameter Tuning:

#### List The Hyperparameter Tuning:

**List The Hyperparameter we used So Far...**

1. Learning Rate
2. Epochs
3. Batches
4. Activation Functions
5. Optimizers
6. Loss Functions
7. Kernel Initializers.

In [1]:
!pip install scikeras


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [100]:
def create_model(learning_rate =0.001,dropout_rate =0.0,activation_function ='relu',init ='glorot_uniform',neuron1 =8, neuron2 =4):
    
    model = Sequential()
    model.add(Dense(neuron1,input_shape=(8,),kernel_initializer=init,activation=activation_function))
    model.add(Dropout(dropout_rate))
    model.add(Dense(neuron2,kernel_initializer=init,activation=activation_function))
    model.add(Dropout(dropout_rate))
    model.add(Dense(1,activation='sigmoid'))

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy',optimizer = optimizer,metrics=['accuracy'])
    return model

In [101]:
model = KerasClassifier(model=create_model,verbose=1)

In [102]:
param_grid = {
              'model__learning_rate':[0.001,0.01],  #__here double under score is important.
              #if its model_lr = just variable naming,but here it is model__learning rate -> Nagative inside object
              'model__dropout_rate':[0.0,0.2],
              'model__activation_function':['relu','tanh'],
              'model__init':['glorot_uniform','he_uniform'],
              'model__neuron1':[8,16],
              'model__neuron2':[4,8],
              'batch_size':[10],
              'epochs':[10]
             }

In [103]:
k_fold = KFold(n_splits=3,shuffle=True, random_state=42)
grid   = GridSearchCV(estimator = model,param_grid =param_grid,cv=k_fold,verbose=10,n_jobs=-1) #parallel Processing.

In [105]:
grid_result = grid.fit(X_train,y_train)

Fitting 3 folds for each of 64 candidates, totalling 192 fits
Epoch 1/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.6792 - loss: 0.6256
Epoch 2/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7231 - loss: 0.5617
Epoch 3/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7248 - loss: 0.5220
Epoch 4/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7394 - loss: 0.5222
Epoch 5/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7508 - loss: 0.5063
Epoch 6/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7410 - loss: 0.5156
Epoch 7/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7296 - loss: 0.5139
Epoch 8/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7557 - loss: 0.4909
Epoch 9/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7296 - loss: 0.5041
Epoch 10/10
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7541 - loss: 0.5035


In [109]:
print("Best: %f using %s"% (grid_result.best_score_,grid_result.best_params_))

means = grid_result.cv_results_['mean_test_score']
stds  = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']

for mean,std,param in zip (means,stds,params):
    print("%f (%f) with: %r" %(mean,std,param))

Best: 0.780121 using {'batch_size': 10, 'epochs': 10, 'model__activation_function': 'relu', 'model__dropout_rate': 0.2, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.01, 'model__neuron1': 8, 'model__neuron2': 4}
0.737773 (0.032288) with: {'batch_size': 10, 'epochs': 10, 'model__activation_function': 'relu', 'model__dropout_rate': 0.0, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.001, 'model__neuron1': 8, 'model__neuron2': 4}
0.734545 (0.007723) with: {'batch_size': 10, 'epochs': 10, 'model__activation_function': 'relu', 'model__dropout_rate': 0.0, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.001, 'model__neuron1': 8, 'model__neuron2': 8}
0.727881 (0.061205) with: {'batch_size': 10, 'epochs': 10, 'model__activation_function': 'relu', 'model__dropout_rate': 0.0, 'model__init': 'glorot_uniform', 'model__learning_rate': 0.001, 'model__neuron1': 16, 'model__neuron2': 4}
0.762171 (0.019005) with: {'batch_size': 10, 'epochs': 10, 'model__activation_fun

In [110]:
results_df = pd.DataFrame(grid_result.cv_results_)
results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_batch_size,param_epochs,param_model__activation_function,param_model__dropout_rate,param_model__init,param_model__learning_rate,param_model__neuron1,param_model__neuron2,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
0,31.581731,2.886544,1.640226,0.317032,10,10,relu,0.0,glorot_uniform,0.001,8,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.780488,0.702439,0.730392,0.737773,0.032288,47
1,32.891334,0.396147,2.236964,0.182808,10,10,relu,0.0,glorot_uniform,0.001,8,8,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.731707,0.726829,0.745098,0.734545,0.007723,48
2,31.744028,1.498702,2.467190,0.247386,10,10,relu,0.0,glorot_uniform,0.001,16,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.795122,0.741463,0.647059,0.727881,0.061205,52
3,31.139967,0.724296,2.685541,0.921289,10,10,relu,0.0,glorot_uniform,0.001,16,8,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.775610,0.775610,0.735294,0.762171,0.019005,18
4,23.565313,1.406881,3.316823,0.734209,10,10,relu,0.0,glorot_uniform,0.010,8,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.795122,0.760976,0.740196,0.765431,0.022644,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,31.789797,0.498287,2.880173,0.999472,10,10,tanh,0.2,he_uniform,0.001,16,8,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.765854,0.726829,0.769608,0.754097,0.019342,31
60,88.277127,1.369003,7.249207,1.889029,10,10,tanh,0.2,he_uniform,0.010,8,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.741463,0.721951,0.759804,0.741073,0.015456,45
61,91.802682,7.321917,4.307855,1.950815,10,10,tanh,0.2,he_uniform,0.010,8,8,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.770732,0.717073,0.769608,0.752471,0.025034,33
62,94.706585,1.276659,2.704799,0.025930,10,10,tanh,0.2,he_uniform,0.010,16,4,"{'batch_size': 10, 'epochs': 10, 'model__activ...",0.760976,0.751220,0.769608,0.760601,0.007512,20
